# LAB- P4: La Elección Óptima de Consumo-Ocio y Ahorro (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P4-v1.0
*   **Capítulo de Referencia**: Capítulo 5, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar la decisión óptima conjunta de consumo-ahorro (decisión intertemporal) y de consumo-ocio (decisión intratemporal que define la oferta de trabajo) a lo largo de un ciclo de vida finito. Estudiar cómo responden el empleo y la acumulación de riqueza a cambios en la productividad salarial y las preferencias de los agentes.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Modelar** conjuntamente decisiones dinámicas intertemporales (ahorro) y estáticas intratemporales (oferta de trabajo).
2.  **Derivar** y comprender la condición de tangencia intratemporal: la tasa marginal de sustitución (TMS) entre consumo y ocio igualada al salario real.
3.  **Visualizar e Interpretar** el efecto renta y el efecto sustitución ante shocks salariales sobre la oferta de trabajo.
4.  **Resolver** sistemas de ecuaciones no lineales acoplados de tamaño $2T \times 2T$ en Python empleando `fsolve` y optimización convexa con `solve_direct_optim`.
5.  **Analizar** la sensibilidad del modelo ante cambios en las preferencias por el ocio (parámetro $\gamma$) y en los rendimientos financieros ($R$).


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Consumo y Ocio
*   **¿Qué estamos haciendo aquí?** Decidiendo cuántas horas trabajar (para tener dinero y consumir) frente a cuántas horas descansar (ocio).
*   **Efecto Sustitución vs Efecto Renta:** Si te suben el sueldo, ¿trabajas más porque cada hora vale más (sustitución) o trabajas menos porque ya eres rico y quieres descansar (renta)?
*   **¡Prueba esto!** Modifica el salario base en el código y observa si el gráfico de horas trabajadas sube o baja.


In [ ]:
# En Google Colab se activarían y descargarían los paquetes necesarios.
# using Pkg; Pkg.activate("."); Pkg.instantiate()


In [ ]:
using Pkg
Pkg.activate("../..")

using MacroAIComp
using Plots
import Plots: mm
using LinearAlgebra
using Interact
using BenchmarkTools


## 1. El Marco Teórico: Consumo, Ahorro y Oferta de Trabajo Endógena

En este modelo, el hogar obtiene satisfacción de dos bienes: el consumo ($C_t$) y el ocio ($O_t$). El ocio representa el tiempo discrecional no dedicado a actividades laborales. Normalizando el tiempo total diario a $1$, la restricción de tiempo es:

$$L_t + O_t = 1$$

Donde $L_t \in [0, 1]$ es la fracción de tiempo dedicada al trabajo. El problema de optimización intertemporal del hogar consiste en maximizar:

$$\max_{\{C_t, L_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t \left[ \gamma \ln(C_t) + (1-\gamma) \ln(1 - L_t) \right]$$

Donde $\gamma \in (0,1)$ representa el peso del consumo en la utilidad (y $1-\gamma$ representa la preferencia relativa por el ocio). 

### 1.1 Restricción Presupuestaria
La acumulación de activos financieros ($B_t$) está determinada por la restricción presupuestaria secuencial:

$$C_t + B_t = W_t L_t + (1+R_{t-1})B_{t-1}$$

Con condiciones inicial y terminal: $B_{-1} = 0$, $B_{T-1} = 0$. Aquí, el ingreso salarial $W_t L_t$ depende endógenamente de las horas trabajadas $L_t$.

### 1.2 Condiciones de Primer Orden (FOC)
El lagrangiano de este problema produce dos condiciones fundamentales:
1.  **Condición Intratemporal (Oferta de Trabajo):**
    $$(1-\gamma) C_t = \gamma (1-L_t) W_t$$
    Esta condición iguala la relación marginal de sustitución entre consumo y ocio con el coste de oportunidad del ocio (el salario real $W_t$). Despejando, la oferta de trabajo óptima es:
    $$L_t = 1 - \left(\frac{1-\gamma}{\gamma}\right)\frac{C_t}{W_t}$$
2.  **Condición Intertemporal (Ecuación de Euler):**
    $$C_{t+1} = \beta (1 + R_t) C_t$$


In [ ]:
params = default_calibration(ConsumptionLeisureParameters)
println("Parámetros Base: ", params)


## 2. Solución Numérica: fsolve vs solve_direct_optim

Para resolver este sistema, implementamos dos solvers:
1.  **`fsolve`**: Resuelve el sistema no lineal acoplado de tamaño $2T \times 2T$ (las $T-1$ ecuaciones de Euler, las $T$ condiciones de oferta de trabajo y la condición terminal de activos en cero).
    *   *Resolución de Errata:* Hemos corregido el error de indexación del código MATLAB del libro (`m5foc.m`), que skipeaba el índice residual $2T-1$ e indexaba un elemento fuera de rango en $2T+1$.
2.  **`solve_direct_optim`**: Resuelve el problema mediante optimización convexa directa (Clarabel).


In [ ]:
# Salario constante exógeno de W = 30
W_base = fill(30.0, params.T)

# Resolución
res_foc = solve_foc_fsolve(params, W_base)
res_opt = solve_direct_optim(params, W_base)

println("Diferencia media en Consumo (FOC vs Optim): ", sum(abs.(res_foc["C"] .- res_opt["C"])) / params.T)
println("Diferencia media en Trabajo (FOC vs Optim): ", sum(abs.(res_foc["L"] .- res_opt["L"])) / params.T)


## 3. Simulación Interactiva en 3 Paneles

Visualizaremos las trayectorias dinámicas resultantes de la simulación en **3 columnas**:
*   **Panel 1 (Consumo e Ingreso Salarial)**: Muestra el consumo óptimo $C_t$ contra el ingreso salarial endógeno $W_t L_t$.
*   **Panel 2 (Oferta de Trabajo y Ocio)**: Muestra la fracción de tiempo dedicada a trabajar $L_t$ y al ocio $O_t = 1 - L_t$.
*   **Panel 3 (Activos Financieros)**: Muestra el perfil de acumulación de riqueza y deuda ($B_t$). Las áreas coloreadas representan periodos de endeudamiento (naranja) o ahorro (azul).

Interactúa con los sliders para analizar los efectos renta y sustitución:
*   `beta`: Paciencia intertemporal.
*   `gamma`: Preferencia por el consumo vs. ocio (cuanto mayor es `gamma`, más valoran el consumo y más horas trabajan).
*   `R`: Tasa de interés de los ahorros.
*   `W`: Salario por unidad de tiempo.


In [ ]:
# Simulación interactiva con Interact.jl
@manipulate for beta_val in 0.90:0.01:0.99, gamma_val in 0.10:0.05:0.90, R_val in -0.05:0.01:0.15, W_val in 10.0:5.0:100.0
    
    params_int = ConsumptionLeisureParameters(30, beta_val, gamma_val, R_val, 0.0)
    W = fill(W_val, params_int.T)
    res = solve_foc_fsolve(params_int, W)
    
    t_axis = 0:(params_int.T - 1)
    
    # Panel 1: Consumo e Ingresos
    p1 = plot(t_axis, res["C"], color=:purple, lw=2.5, label="Consumo (C)")
    plot!(t_axis, res["W_L"], color=:forestgreen, lw=2.5, ls=:dash, label="Ingreso (W·L)")
    title!("Consumo e Ingreso Salarial")
    xlabel!("Periodo (t)")
    ylabel!("Bienes")
    
    # Panel 2: Oferta de Trabajo y Ocio
    p2 = plot(t_axis, res["L"], color=:red, lw=2.5, label="Trabajo (L)")
    plot!(t_axis, res["O"], color=:teal, lw=2.5, ls=:dot, label="Ocio (O=1-L)")
    ylims!(-0.05, 1.05)
    title!("Asignación del Tiempo")
    xlabel!("Periodo (t)")
    ylabel!("Fracción de Tiempo")
    
    # Panel 3: Activos
    p3 = plot(t_axis, res["B"], color=:steelblue, lw=2.5, label="Activos (B)")
    hline!([0.0], color=:black, ls=:dot, label="")
    plot!(t_axis, max.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color=:steelblue, lw=0, label="Acreedor")
    plot!(t_axis, min.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color=:orange, lw=0, label="Deudor")
    title!("Evolución de Activos")
    xlabel!("Periodo (t)")
    ylabel!("Riqueza Neta")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Decisión de Consumo-Ocio Intertemporal", top_margin=10mm)
end


## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones analíticas en tu Cuaderno de Bitácora tras interactuar con la simulación:

1.  **El Efecto Renta y Sustitución sobre el Trabajo**:
    *   Incrementa el salario exógeno $W$ de 30 a 60. ¿Cómo reacciona la oferta de trabajo inicial $L_0$?
    *   En la teoría microeconómica, un aumento de salario tiene dos efectos contrapuestos sobre el ocio: el **efecto sustitución** (el ocio es más caro, por lo que trabajo más) y el **efecto renta** (soy más rico, por lo que consumo más ocio). Dado que con utilidad logarítmica estos efectos se cancelan exactamente en el periodo inicial si la riqueza es constante, describe cómo cambia el perfil del trabajo a lo largo del ciclo vital y por qué.
2.  **El Peso de las Preferencias ($\gamma$)**:
    *   Establece $\gamma = 0.20$ (alta preferencia por el ocio) y luego $\gamma = 0.80$ (alta preferencia por consumir bienes). 
    *   Compara los niveles de empleo resultantes en el Panel 2. ¿Cómo se ajustan las curvas de consumo e ingresos salariales en el Panel 1 en respuesta a esta variación en la psicología del consumidor?
3.  **La Dinámica del Ahorro y la Tasa de Interés**:
    *   Fija $R = 10\%$ y describe la trayectoria de acumulación de activos financieros. ¿El individuo es deudor o acreedor la mayor parte de su vida?
    *   ¿Por qué el individuo reduce drásticamente su ocio (trabaja más) al principio de su ciclo de vida cuando la tasa de interés es muy elevada? Explica la ganancia intertemporal de ahorrar capital temprano.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Aislamiento Paramétrico**: Las rutinas matemáticas y de simulación están completamente desacopladas de la interfaz visual, residiendo en `src/macroaicomp/models/consumption_leisure.jl`.
2.  **Control de Calidad Automático**: Se ha verificado que la simulación es robusta mediante tests unitarios automatizados (`Test.jl`).
3.  **Control de Versiones Limpio**: Este cuaderno ha sido limpiado de metadatos volátiles mediante `nbstripout` antes de guardarse en el repositorio.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation
@btime solve_foc_fsolve($params, $W_base)
